# Fork-2 front-end runner (Colab)

**Thin runner only.** All pipeline logic lives in the git-tracked `scripts/fork2_01_typing_floors.R` (harmonize → ONE FlowSOM cross-cohort cell-typing → Gate B canary → Gate A floors). This notebook just: mounts Drive, points the script at the Drive data paths, installs the Bioconductor IMC stack (reachable on Colab, unlike the local box), runs the script, and prints **R1–R4** for review. It **stops after Gate A** — no Gate-C proximity feature.

### Before you run
1. **Move the 3 data files to Drive** per the manifest (see the chat message / `docs`):
   - `→ /content/drive/MyDrive/penumbra/data/meyer/sce_ALL_sub.rds` (84 MB)
   - `→ /content/drive/MyDrive/penumbra/data/metabric/SingleCells.fst` (326 MB)
   - `→ /content/drive/MyDrive/penumbra/data/metabric/IMCClinical.fst` (12 KB; not used until Gate D, but stage it now)
2. **Push the latest repo commits to GitHub** (`github.com/hcl1216/penumbra`) — this notebook clones the code from there, not from Drive.

### Runtime / RAM
- **High-RAM runtime recommended.** Reads a 341 MB `.fst` and builds a ~1.21M×17 matrix; the FlowSOM **SOM is trained on a 200k balanced cross-cohort subsample** then mapped to all ~1.21M cells (no full-set clustering → no OOM). Standard 12–13 GB usually works; High-RAM is the safe choice.
- Expect ~10–20 min Bioconductor install + ~10–20 min pipeline.

### Repo access — FLAG (confirm before running)
I could **not verify whether `github.com/hcl1216/penumbra` is public or private** (the local repo has no configured remote). 
- **PUBLIC** → the clone cell works as-is.
- **PRIVATE** → set `GITHUB_PAT` in the clone cell (a fine-grained token with read access). Don't commit the token.

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/penumbra'
import os
for p in [f'{DRIVE}/data/meyer/sce_ALL_sub.rds',
          f'{DRIVE}/data/metabric/SingleCells.fst',
          f'{DRIVE}/data/metabric/IMCClinical.fst']:
    print(('OK   ' if os.path.exists(p) else 'MISSING '), p)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

In [ ]:
# 2) Get the code from git (logic comes from the repo, NOT Drive)
GITHUB_PAT = ''   # leave '' if the repo is PUBLIC; paste a read PAT if PRIVATE
REPO_URL = 'github.com/hcl1216/penumbra.git'
url = f'https://{GITHUB_PAT + "@" if GITHUB_PAT else ""}{REPO_URL}'
import os
if os.path.isdir('/content/penumbra/.git'):
    !cd /content/penumbra && git pull --ff-only
else:
    !git clone {url} /content/penumbra
REPO = '/content/penumbra'
assert os.path.exists(f'{REPO}/scripts/fork2_01_typing_floors.R'), 'script not found — check repo/branch'
print('repo at', REPO)

In [ ]:
# 3) Point the script at the Drive data + results paths (read by the R script via env vars)
import os
os.environ['MEYER_SCE']            = f'{DRIVE}/data/meyer/sce_ALL_sub.rds'
os.environ['METABRIC_SC']          = f'{DRIVE}/data/metabric/SingleCells.fst'
os.environ['PENUMBRA_RESULTS_DIR'] = f'{DRIVE}/results'
print({k: os.environ[k] for k in ['MEYER_SCE','METABRIC_SC','PENUMBRA_RESULTS_DIR']})

In [ ]:
%%bash
# 4) Install the Bioconductor IMC stack + CRAN deps (bioconductor.org IS reachable on Colab).
#    System R + Rscript are present on Colab. This is the slow cell (~10-20 min).
Rscript -e '
  options(repos=c(CRAN="https://cloud.r-project.org"), timeout=1200)
  if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
  for (p in c("fst","survival")) if (!requireNamespace(p,quietly=TRUE)) install.packages(p)
  bioc <- c("SingleCellExperiment","FlowSOM","imcRtools","cytomapper")
  need <- bioc[!sapply(bioc, requireNamespace, quietly=TRUE)]
  if (length(need)) BiocManager::install(need, update=FALSE, ask=FALSE)
  cat("installed:", paste(sapply(c(bioc,"fst","survival"), requireNamespace, quietly=TRUE), collapse=" "), "\n")'


In [ ]:
# 5) Run the git-tracked R script via Rscript (inherits the env vars set above; --file gives REPO)
!cd /content/penumbra && Rscript scripts/fork2_01_typing_floors.R

In [ ]:
# 6) Print R1-R4 (persisted to Drive results/) for review -> paste these back before Gate C
for r in ['fork2_R1_marker_status.md','fork2_R2_celltypes.md','fork2_R3_canary.md','fork2_R4_floors.md']:
    p = f'{DRIVE}/results/{r}'
    print('\n' + '='*78 + f'\n{r}\n' + '='*78)
    print(open(p).read() if os.path.exists(p) else '(missing — the run did not reach this artifact)')